# Extract residual-stream activations — Qwen2.5-1.5B (Colab)

Runs on Colab GPU. Extracts `resid_post` activations at the **last token**
for every statement in the Geometry of Truth datasets and saves them to
Google Drive as `.npy` files, to be pulled down and probed locally on CPU.

Conventions (see project `CLAUDE.md`):
- Activations shape `[n_statements, n_layers, d_model]`.
- Model is the **base** (non-instruct) Qwen2.5-1.5B, loaded via HF handoff
  into TransformerLens.
- Sign convention (positive = true/honest) is applied later, at probing time.

## 1. Mount Drive and set the save path

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# All activations/labels get written here, then pulled down to
# data/activations/ locally for the (CPU-only) probing side of the project.
SAVE_DIR = "/content/drive/MyDrive/bluedot_project/"
import os; os.makedirs(SAVE_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install dependencies

In [ ]:
# transformer_lens gives cache access to resid_post at every layer in one
# forward pass; scikit-learn/pandas/matplotlib are for the local probing side.
!pip install transformer_lens scikit-learn pandas matplotlib


## 3. Imports

In [ ]:
import torch
from transformer_lens import HookedTransformer
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"


## 4. Clone the Geometry of Truth datasets

In [ ]:
!git clone https://github.com/saprmarks/geometry-of-truth.git
import pandas as pd

# Sanity check: columns are `statement` and `label` (1 = true), per CLAUDE.md.
cities = pd.read_csv("geometry-of-truth/datasets/cities.csv")
print(cities[["statement", "label"]].head(10)); print(len(cities))


fatal: destination path 'geometry-of-truth' already exists and is not an empty directory.
                                        statement  label
0             The city of Krasnodar is in Russia.      1
1       The city of Krasnodar is in South Africa.      0
2                  The city of Lodz is in Poland.      1
3  The city of Lodz is in the Dominican Republic.      0
4            The city of Maracay is in Venezuela.      1
5                The city of Maracay is in China.      0
6              The city of Baku is in Azerbaijan.      1
7                 The city of Baku is in Ukraine.      0
8                  The city of Baoji is in China.      1
9              The city of Baoji is in Guatemala.      0
1496


## 5. Activation extraction helper

In [ ]:
import numpy as np
from tqdm import tqdm

def extract_acts(model, texts):
    """Run each statement through the model and grab resid_post at the last token, every layer."""
    n_layers, d_model = model.cfg.n_layers, model.cfg.d_model
    acts = np.zeros((len(texts), n_layers, d_model), dtype=np.float32)
    with torch.no_grad():
        for i, t in enumerate(tqdm(texts)):
            # One statement at a time (not batched) to keep peak memory low
            # on the T4; caching every resid_post is otherwise expensive.
            _, cache = model.run_with_cache(
                t, names_filter=lambda n: "resid_post" in n)
            for L in range(n_layers):
                # [-1] = last token position, i.e. the final token of the statement.
                acts[i, L] = cache["resid_post", L][0, -1, :].float().cpu().numpy()
            del cache  # avoid accumulating cached activations across statements
    return acts


## 6. Load the model — base Qwen2.5-1.5B via HF handoff

In [ ]:
from transformers import AutoModelForCausalLM

# Load the base (non-instruct) checkpoint in fp16 to fit comfortably on a T4.
hf_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B", torch_dtype=torch.float16, low_cpu_mem_usage=True
)
# from_pretrained_no_processing hands the HF weights straight to TransformerLens
# without folding LayerNorm/etc., so resid_post matches the original model exactly.
model = HookedTransformer.from_pretrained_no_processing(
    "qwen2.5-1.5b", hf_model=hf_model, dtype=torch.float16, device=device
)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded pretrained model qwen2.5-1.5b into HookedTransformer


## 7. Extract and save activations for each dataset

In [ ]:
# cities / neg_cities / sp_en_trans: matched true/false statement pairs used
# for the contrast-pair direction comparison. Saved as <name>_acts.npy /
# <name>_labels.npy so the local probing code can load them straight from
# data/activations/.
for name in ["cities", "neg_cities", "sp_en_trans"]:
    df = pd.read_csv(f"geometry-of-truth/datasets/{name}.csv")
    acts = extract_acts(model, df["statement"].tolist())
    np.save(SAVE_DIR + f"{name}_acts.npy", acts)
    np.save(SAVE_DIR + f"{name}_labels.npy", df["label"].to_numpy())


100%|██████████| 354/354 [00:48<00:00,  7.36it/s]
